# IPPO on `simple_tag_v3`

Independent PPO (IPPO) baseline for the PettingZoo / MPE2 `simple_tag_v3` multi-agent environment.

## Algorithm overview
- **IPPO**: each agent learns from its own local observation with standard PPO — no centralised critic
- **Role-based policy sharing**: adversaries share one actor-critic; good agents share another (they do **not** share across roles)
- **Environment interface**: `parallel_env()` — all agents act simultaneously each step
- **Discrete actions**: `0=no_action  1=left  2=right  3=down  4=up`

## Notebook structure
1. Imports, Config, seed utilities
2. Environment helpers
3. Actor-Critic network (MLP, hidden=128, Tanh)
4. PPO Rollout Buffer (per-agent GAE, then merged)
5. IPPO class (two policies + two buffers + PPO update)
6. Training loop
7. Evaluation (deterministic)
8. Training curves & checkpoint loading

In [1]:
import os
import random
import time
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt

from mpe2 import simple_tag_v3


# ── Hyperparameters ────────────────────────────────────────────────────────────

@dataclass
class Config:
    # --- Environment ---
    max_cycles: int = 100       # episode length (mpe2 default is 25)

    # --- Training schedule ---
    total_timesteps: int = 3_000_000
    rollout_steps: int = 2048   # env steps collected before each PPO update

    # --- PPO ---
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_eps: float = 0.1       # keep ≤ 0.15 for stability
    entropy_coef: float = 0.01
    value_coef: float = 0.5
    max_grad_norm: float = 0.5
    ppo_epochs: int = 5         # few epochs avoids stale-data instability
    num_minibatches: int = 1    # 1 or at most 2
    advantage_normalization: bool = True
    value_normalization: bool = True   # normalise value targets before critic loss

    # --- Network ---
    hidden_size: int = 128
    lr: float = 3e-4

    # --- Reproducibility ---
    seed: int = 42
    device: str = "cpu"         # "cuda" if GPU available

    # --- Logging / saving ---
    log_interval: int = 10          # print every N PPO updates
    eval_interval: int = 50
    eval_episodes: int = 10
    save_dir: str = "models"
    checkpoint_interval: int = 100


def set_seed(seed: int):
    """Set Python / NumPy / PyTorch seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

C:\Users\17518\PycharmProjects\COMP579 Final project\.venv312\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
import sys, torch
print("Python     :", sys.executable)
print("PyTorch    :", torch.__version__)
print("CUDA built :", torch.version.cuda)          # None = CPU-only build
print("CUDA avail :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))

Python     : C:\Users\17518\PycharmProjects\COMP579 Final project\.venv312\Scripts\python.exe
PyTorch    : 2.11.0+cu126
CUDA built : 12.6
CUDA avail : True
GPU        : NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
# ── Environment helpers ────────────────────────────────────────────────────────

def make_env(max_cycles: int = 100, render_mode=None):
    """
    Create simple_tag_v3 parallel env with default settings:
      3 adversaries (predators), 1 good agent (prey), 2 obstacles, discrete actions.
    """
    return simple_tag_v3.parallel_env(
        num_good=1,
        num_adversaries=3,
        num_obstacles=2,
        max_cycles=max_cycles,
        continuous_actions=False,
        render_mode=render_mode,
    )


def get_agent_roles(agents: list) -> dict:
    """
    Map agent name → role string.
      'adversary_0', 'adversary_1', ... → 'adversary'
      'agent_0', 'agent_1', ...          → 'good'
    """
    return {
        a: "adversary" if a.startswith("adversary") else "good"
        for a in agents
    }


# ── Explore the environment ────────────────────────────────────────────────────
_env = make_env()
obs_dict, _ = _env.reset(seed=0)

_roles            = get_agent_roles(_env.agents)
_adversary_agents = [a for a in _env.agents if _roles[a] == "adversary"]
_good_agents      = [a for a in _env.agents if _roles[a] == "good"]

print(f"All agents        : {_env.agents}")
print(f"Adversary agents  : {_adversary_agents}")
print(f"Good agents       : {_good_agents}")
print(f"Adversary obs_dim : {_env.observation_space(_adversary_agents[0]).shape[0]}")
print(f"Good obs_dim      : {_env.observation_space(_good_agents[0]).shape[0]}")
print(f"Action dim        : {_env.action_space(_adversary_agents[0]).n}")
print("Actions           :  0=no_action  1=left  2=right  3=down  4=up")
_env.close()

All agents        : ['adversary_0', 'adversary_1', 'adversary_2', 'agent_0']
Adversary agents  : ['adversary_0', 'adversary_1', 'adversary_2']
Good agents       : ['agent_0']
Adversary obs_dim : 16
Good obs_dim      : 14
Action dim        : 5
Actions           :  0=no_action  1=left  2=right  3=down  4=up


In [4]:
# ── Actor-Critic Network ───────────────────────────────────────────────────────
# Each role gets its OWN ActorCritic instance (adversaries and good agents
# do NOT share weights). Critic uses local observation only — pure IPPO,
# no centralised value function.

def build_mlp(input_dim: int, hidden_size: int, output_dim: int) -> nn.Sequential:
    """Two-hidden-layer MLP with Tanh activations."""
    return nn.Sequential(
        nn.Linear(input_dim, hidden_size), nn.Tanh(),
        nn.Linear(hidden_size, hidden_size), nn.Tanh(),
        nn.Linear(hidden_size, output_dim),
    )


class ActorCritic(nn.Module):
    """
    Separate actor and critic MLPs for one policy role.

    Actor : obs (local) → logits over discrete actions  shape (batch, action_dim)
    Critic: obs (local) → scalar state value            shape (batch,)
    """

    def __init__(self, obs_dim: int, action_dim: int, hidden_size: int = 128):
        super().__init__()
        self.actor  = build_mlp(obs_dim, hidden_size, action_dim)
        self.critic = build_mlp(obs_dim, hidden_size, 1)

    def get_value(self, obs: torch.Tensor) -> torch.Tensor:
        """obs: (batch, obs_dim) → value: (batch,)"""
        return self.critic(obs).squeeze(-1)

    def get_action_and_value(self, obs: torch.Tensor, action: torch.Tensor = None):
        """
        Sample or evaluate an action.

        Args:
            obs:    (batch, obs_dim)
            action: (batch,)  optional — if given, evaluate log_prob of that action

        Returns:
            action, log_prob, entropy, value  — each shape (batch,)
        """
        logits = self.actor(obs)
        dist   = Categorical(logits=logits)
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), self.critic(obs).squeeze(-1)

    def get_deterministic_action(self, obs: torch.Tensor) -> torch.Tensor:
        """Argmax (greedy) action for evaluation."""
        return self.actor(obs).argmax(dim=-1)

In [5]:
class PPORolloutBuffer:
    """
    Rollout buffer for one policy role (adversary or good agent).

    Stores per-agent trajectory lists separately so that GAE can be computed
    along each agent's own time-series. After GAE, all agent data are merged
    into a flat dataset for PPO minibatch updates.

    Usage:
        reset()
        → store(agent, …) × T steps  (for each agent at each env step)
        → compute_returns_and_advantages(last_values, last_dones, gamma, gae_lambda)
        → get_batches(num_minibatches)  [inside PPO epoch loop]
    """

    def __init__(self, agent_names: List[str], obs_dim: int, device: str = "cpu"):
        self.agent_names = agent_names
        self.obs_dim     = obs_dim
        self.device      = device
        self.reset()

    def reset(self):
        """Clear stored data. Call before each rollout."""
        self.obs       = {a: [] for a in self.agent_names}
        self.actions   = {a: [] for a in self.agent_names}
        self.log_probs = {a: [] for a in self.agent_names}
        self.rewards   = {a: [] for a in self.agent_names}
        self.dones     = {a: [] for a in self.agent_names}
        self.values    = {a: [] for a in self.agent_names}
        # Populated by compute_returns_and_advantages()
        self._obs_t = self._act_t = self._lp_t = self._ret_t = self._adv_t = None

    def store(
        self, agent: str,
        obs: np.ndarray, action: int, log_prob: float,
        reward: float, done: bool, value: float,
    ):
        """Append one transition for the given agent."""
        self.obs[agent].append(obs)
        self.actions[agent].append(action)
        self.log_probs[agent].append(log_prob)
        self.rewards[agent].append(reward)
        self.dones[agent].append(float(done))
        self.values[agent].append(value)

    def compute_returns_and_advantages(
        self,
        last_values: Dict[str, float],
        last_dones:  Dict[str, bool],
        gamma: float,
        gae_lambda: float,
    ):
        """
        Compute GAE advantages per agent, then concatenate into flat tensors
        ready for PPO minibatch updates.

        Args:
            last_values : {agent: float}  bootstrap V(s_T) for each agent
            last_dones  : {agent: bool}   whether each agent's episode ended at T
        """
        all_obs, all_act, all_lp, all_ret, all_adv = [], [], [], [], []

        for agent in self.agent_names:
            rewards = np.array(self.rewards[agent], dtype=np.float32)
            dones   = np.array(self.dones[agent],   dtype=np.float32)
            values  = np.array(self.values[agent],  dtype=np.float32)
            T = len(rewards)
            if T == 0:
                continue

            advantages = np.zeros(T, dtype=np.float32)
            last_gae   = 0.0

            for t in reversed(range(T)):
                if t == T - 1:
                    # Bootstrap from last observation
                    next_nt  = 1.0 - float(last_dones.get(agent, False))
                    next_val = float(last_values.get(agent, 0.0))
                else:
                    next_nt  = 1.0 - dones[t + 1]
                    next_val = float(values[t + 1])

                delta      = rewards[t] + gamma * next_val * next_nt - values[t]
                last_gae   = delta + gamma * gae_lambda * next_nt * last_gae
                advantages[t] = last_gae

            returns = advantages + values   # TD(λ) returns

            all_obs.append(np.array(self.obs[agent],       dtype=np.float32))
            all_act.append(np.array(self.actions[agent],   dtype=np.int64))
            all_lp.append( np.array(self.log_probs[agent], dtype=np.float32))
            all_ret.append(returns)
            all_adv.append(advantages)

        # Merge all agents → flat tensors
        self._obs_t = torch.FloatTensor(np.concatenate(all_obs)).to(self.device)
        self._act_t = torch.LongTensor( np.concatenate(all_act)).to(self.device)
        self._lp_t  = torch.FloatTensor(np.concatenate(all_lp )).to(self.device)
        self._ret_t = torch.FloatTensor(np.concatenate(all_ret)).to(self.device)
        self._adv_t = torch.FloatTensor(np.concatenate(all_adv)).to(self.device)

    @property
    def size(self) -> int:
        """Total number of stored transitions across all agents."""
        return sum(len(v) for v in self.obs.values())

    def get_batches(self, num_minibatches: int):
        """
        Yield shuffled minibatches of (obs, actions, log_probs, returns, advantages).
        With num_minibatches=1, the entire buffer is one batch.
        """
        assert self._obs_t is not None, "Call compute_returns_and_advantages() first."
        N          = self._obs_t.shape[0]
        idx        = torch.randperm(N)
        batch_size = N // num_minibatches
        for start in range(0, N, batch_size):
            b = idx[start : start + batch_size]
            yield (
                self._obs_t[b], self._act_t[b], self._lp_t[b],
                self._ret_t[b], self._adv_t[b],
            )

In [6]:
class IPPO:
    """
    Independent PPO for simple_tag_v3.

    Maintains two separate actor-critic policies:
      adversary_policy — shared by all adversary agents (3 by default)
      good_policy      — shared by all good agents (1 by default)

    Each policy is updated independently with standard PPO + GAE.
    Critic input is the agent's LOCAL observation only (no centralised value fn).

    Policy routing:
        ippo.act(agent, obs)           → routes to the correct role policy
        ippo.store_transition(agent, …)→ routes to the correct role buffer
        ippo.update(last_obs, last_dones) → GAE + PPO for both roles
    """

    ROLES = ("adversary", "good")

    def __init__(
        self,
        adversary_agents: List[str],
        good_agents: List[str],
        adv_obs_dim: int,
        good_obs_dim: int,
        action_dim: int,
        cfg: Config,
    ):
        self.cfg    = cfg
        self.device = torch.device(cfg.device)

        self.adversary_agents = adversary_agents
        self.good_agents      = good_agents

        # Map every agent name → role string
        self._role: Dict[str, str] = (
            {a: "adversary" for a in adversary_agents} |
            {a: "good"      for a in good_agents}
        )

        # One ActorCritic per role — separate input dims, separate weights
        self.policies: Dict[str, ActorCritic] = {
            "adversary": ActorCritic(adv_obs_dim,  action_dim, cfg.hidden_size).to(self.device),
            "good":      ActorCritic(good_obs_dim, action_dim, cfg.hidden_size).to(self.device),
        }
        self.optimizers = {
            role: optim.Adam(pol.parameters(), lr=cfg.lr)
            for role, pol in self.policies.items()
        }
        # One rollout buffer per role
        self.buffers: Dict[str, PPORolloutBuffer] = {
            "adversary": PPORolloutBuffer(adversary_agents, adv_obs_dim,  cfg.device),
            "good":      PPORolloutBuffer(good_agents,      good_obs_dim, cfg.device),
        }
        self.update_count = 0

    # ── Routing helpers ───────────────────────────────────────────────────────
    def role_of(self, agent: str)    -> str:              return self._role[agent]
    def policy_of(self, agent: str)  -> ActorCritic:      return self.policies[self.role_of(agent)]
    def buffer_of(self, agent: str)  -> PPORolloutBuffer: return self.buffers[self.role_of(agent)]

    # ── Inference ─────────────────────────────────────────────────────────────
    @torch.no_grad()
    def act(self, agent: str, obs: np.ndarray):
        """Stochastic action for training. Returns (action, log_prob, value)."""
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(self.device)  # (1, obs_dim)
        a, lp, _, v = self.policy_of(agent).get_action_and_value(obs_t)
        return a.item(), lp.item(), v.item()

    @torch.no_grad()
    def act_deterministic(self, agent: str, obs: np.ndarray) -> int:
        """Argmax action for evaluation (no exploration)."""
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(self.device)
        return self.policy_of(agent).get_deterministic_action(obs_t).item()

    @torch.no_grad()
    def get_value(self, agent: str, obs: np.ndarray) -> float:
        """Bootstrap V(s) for GAE."""
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(self.device)
        return self.policy_of(agent).get_value(obs_t).item()

    # ── Buffer interface ───────────────────────────────────────────────────────
    def store_transition(self, agent, obs, action, log_prob, reward, done, value):
        """Route transition to the correct role buffer."""
        self.buffer_of(agent).store(agent, obs, action, log_prob, reward, done, value)

    def reset_buffers(self):
        for buf in self.buffers.values():
            buf.reset()

    # ── PPO update ────────────────────────────────────────────────────────────
    def update(self, last_obs: dict, last_dones: dict) -> dict:
        """
        Compute GAE for both roles, run PPO updates, clear buffers.

        Args:
            last_obs   : {agent: obs_array}  — observations at end of rollout
            last_dones : {agent: bool}        — whether episode ended at rollout end

        Returns:
            {role: {"pg_loss", "value_loss", "entropy", "approx_kl"}}
        """
        last_values = {a: self.get_value(a, obs) for a, obs in last_obs.items()}

        # Compute GAE per role
        for role in self.ROLES:
            buf     = self.buffers[role]
            members = self.adversary_agents if role == "adversary" else self.good_agents
            lv = {a: last_values.get(a, 0.0)  for a in members}
            ld = {a: last_dones.get(a,  False) for a in members}
            if buf.size > 0:
                buf.compute_returns_and_advantages(lv, ld, self.cfg.gamma, self.cfg.gae_lambda)

        # PPO update per role
        metrics = {}
        for role in self.ROLES:
            buf = self.buffers[role]
            if buf.size == 0:
                continue
            metrics[role] = self._ppo_update(
                self.policies[role], self.optimizers[role], buf
            )

        self.update_count += 1
        self.reset_buffers()
        return metrics

    def _ppo_update(self, policy: ActorCritic, optimizer, buf: PPORolloutBuffer) -> dict:
        """One full PPO update pass (ppo_epochs × num_minibatches) for a single role."""
        cfg = self.cfg

        # Advantage normalisation over the full role batch
        adv = buf._adv_t
        if cfg.advantage_normalization:
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        # Optional value-target normalisation for critic stability
        ret = buf._ret_t
        if cfg.value_normalization:
            ret = (ret - ret.mean()) / (ret.std() + 1e-8)

        # Temporarily patch buffer with normalised tensors
        orig_adv, orig_ret = buf._adv_t, buf._ret_t
        buf._adv_t, buf._ret_t = adv, ret

        pg_losses, v_losses, entropies, kls = [], [], [], []

        for _ in range(cfg.ppo_epochs):
            for obs_b, act_b, old_lp_b, ret_b, adv_b in buf.get_batches(cfg.num_minibatches):
                _, new_lp, entropy, new_val = policy.get_action_and_value(obs_b, act_b)

                # Clipped PPO policy loss
                log_ratio = new_lp - old_lp_b
                ratio     = log_ratio.exp()
                # Approx KL for monitoring — should stay small with clip_eps=0.1
                approx_kl = ((ratio - 1) - log_ratio).mean().item()

                pg_loss = torch.max(
                    -adv_b * ratio,
                    -adv_b * ratio.clamp(1 - cfg.clip_eps, 1 + cfg.clip_eps),
                ).mean()

                v_loss       = 0.5 * (new_val - ret_b).pow(2).mean()
                entropy_loss = entropy.mean()
                loss = pg_loss + cfg.value_coef * v_loss - cfg.entropy_coef * entropy_loss

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), cfg.max_grad_norm)
                optimizer.step()

                pg_losses.append(pg_loss.item())
                v_losses.append(v_loss.item())
                entropies.append(entropy_loss.item())
                kls.append(approx_kl)

        buf._adv_t, buf._ret_t = orig_adv, orig_ret  # restore unnormalised tensors
        return {
            "pg_loss":    float(np.mean(pg_losses)),
            "value_loss": float(np.mean(v_losses)),
            "entropy":    float(np.mean(entropies)),
            "approx_kl":  float(np.mean(kls)),
        }

    # ── Checkpointing ─────────────────────────────────────────────────────────
    def save(self, path: str):
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        torch.save({
            "adversary_policy": self.policies["adversary"].state_dict(),
            "good_policy":      self.policies["good"].state_dict(),
            "adversary_optim":  self.optimizers["adversary"].state_dict(),
            "good_optim":       self.optimizers["good"].state_dict(),
            "update_count":     self.update_count,
        }, path)
        print(f"Checkpoint saved → {path}")

    def load(self, path: str):
        ckpt = torch.load(path, map_location=self.device)
        self.policies["adversary"].load_state_dict(ckpt["adversary_policy"])
        self.policies["good"].load_state_dict(ckpt["good_policy"])
        self.optimizers["adversary"].load_state_dict(ckpt["adversary_optim"])
        self.optimizers["good"].load_state_dict(ckpt["good_optim"])
        self.update_count = ckpt.get("update_count", 0)
        print(f"Checkpoint loaded ← {path}  (update {self.update_count})")

In [7]:
def train(cfg: Config, resume_from: str = None):
    """
    IPPO training loop for simple_tag_v3.

    Collects `rollout_steps` environment steps into role-specific buffers,
    computes GAE advantages, then updates both role policies with PPO.
    Repeats until `total_timesteps` is reached.

    Multi-env extension note:
        To support num_envs > 1, replace the single `env` with a list of
        env instances. In the rollout collection block, loop over each env
        and append transitions to the same role buffers. IPPO.update() and
        PPORolloutBuffer are already compatible with data from multiple envs;
        only the inner rollout loop needs to change.

    Args:
        cfg:         Config dataclass with all hyperparameters.
        resume_from: optional checkpoint path to resume training from.

    Returns:
        ippo    : trained IPPO instance
        history : dict of logged metrics over time
    """
    set_seed(cfg.seed)
    os.makedirs(cfg.save_dir, exist_ok=True)

    # ── Environment ───────────────────────────────────────────────────────────
    env = make_env(max_cycles=cfg.max_cycles)
    obs_dict, _ = env.reset(seed=cfg.seed)

    all_agents       = env.agents
    roles            = get_agent_roles(all_agents)
    adversary_agents = [a for a in all_agents if roles[a] == "adversary"]
    good_agents      = [a for a in all_agents if roles[a] == "good"]

    adv_obs_dim  = env.observation_space(adversary_agents[0]).shape[0]
    good_obs_dim = env.observation_space(good_agents[0]).shape[0]
    action_dim   = env.action_space(adversary_agents[0]).n

    # ── Agent ─────────────────────────────────────────────────────────────────
    ippo = IPPO(adversary_agents, good_agents,
                adv_obs_dim, good_obs_dim, action_dim, cfg)
    if resume_from:
        ippo.load(resume_from)

    # ── Logging state ─────────────────────────────────────────────────────────
    history: Dict[str, list] = {
        "steps": [], "update": [],
        "adv_return": [], "good_return": [],
        "adv_pg_loss": [], "adv_v_loss": [], "adv_entropy": [],
        "good_pg_loss": [], "good_v_loss": [], "good_entropy": [],
    }
    ep_returns:  Dict[str, list]  = {"adversary": [], "good": []}
    ep_acc:      Dict[str, float] = {a: 0.0 for a in all_agents}
    total_steps  = 0
    update_count = 0
    start_time   = time.time()

    print(f"Training IPPO on simple_tag_v3")
    print(f"  adv_obs={adv_obs_dim}  good_obs={good_obs_dim}  actions={action_dim}")
    print(f"  total_timesteps={cfg.total_timesteps:,}  rollout_steps={cfg.rollout_steps}")
    print(f"  ppo_epochs={cfg.ppo_epochs}  num_minibatches={cfg.num_minibatches}  clip_eps={cfg.clip_eps}")
    print()

    # ── Training loop ─────────────────────────────────────────────────────────
    while total_steps < cfg.total_timesteps:
        ippo.reset_buffers()
        steps_this_rollout = 0

        # ── Rollout collection ────────────────────────────────────────────────
        while steps_this_rollout < cfg.rollout_steps:
            # Episode boundary: reset if all agents finished
            if not env.agents:
                for a, ret in ep_acc.items():
                    ep_returns[roles[a]].append(ret)
                ep_acc      = {a: 0.0 for a in all_agents}
                obs_dict, _ = env.reset()

            active = list(obs_dict.keys())
            actions: Dict[str, int]   = {}
            log_probs: Dict[str, float] = {}
            values: Dict[str, float]    = {}

            for agent in active:
                a, lp, v        = ippo.act(agent, obs_dict[agent])
                actions[agent]  = a
                log_probs[agent]= lp
                values[agent]   = v

            next_obs, rewards, terms, truncs, _ = env.step(actions)
            dones = {
                a: terms.get(a, False) or truncs.get(a, False)
                for a in active
            }

            for agent in active:
                ippo.store_transition(
                    agent, obs_dict[agent],
                    actions[agent], log_probs[agent],
                    rewards.get(agent, 0.0), dones[agent], values[agent],
                )
                ep_acc[agent] = ep_acc.get(agent, 0.0) + rewards.get(agent, 0.0)

            obs_dict            = next_obs
            steps_this_rollout += 1
            total_steps        += 1

        # ── Bootstrap V(s_T) for GAE ──────────────────────────────────────────
        last_obs:   Dict[str, np.ndarray] = {}
        last_dones: Dict[str, bool]       = {}
        for a in all_agents:
            if a in obs_dict:
                last_obs[a]   = obs_dict[a]
                last_dones[a] = False
            else:
                # Agent's episode ended before rollout finished; bootstrap = 0
                dim           = adv_obs_dim if roles[a] == "adversary" else good_obs_dim
                last_obs[a]   = np.zeros(dim, dtype=np.float32)
                last_dones[a] = True

        # ── PPO update ────────────────────────────────────────────────────────
        metrics      = ippo.update(last_obs, last_dones)
        update_count += 1

        # ── Logging ───────────────────────────────────────────────────────────
        if update_count % cfg.log_interval == 0 and ep_returns["adversary"]:
            elapsed   = time.time() - start_time
            fps       = total_steps / max(elapsed, 1e-6)
            mean_adv  = float(np.mean(ep_returns["adversary"][-20:]))
            mean_good = float(np.mean(ep_returns["good"][-20:]))
            adv_m     = metrics.get("adversary", {})
            good_m    = metrics.get("good", {})

            print(
                f"Update {update_count:5d} | Steps {total_steps:9,d} | FPS {fps:5.0f} | "
                f"Adv {mean_adv:+6.2f} | Good {mean_good:+6.2f} | "
                f"adv[pg={adv_m.get('pg_loss', 0):+.3f} "
                f"v={adv_m.get('value_loss', 0):.3f} "
                f"ent={adv_m.get('entropy', 0):.3f} "
                f"kl={adv_m.get('approx_kl', 0):.4f}] "
                f"good[pg={good_m.get('pg_loss', 0):+.3f}]"
            )
            history["steps"].append(total_steps)
            history["update"].append(update_count)
            history["adv_return"].append(mean_adv)
            history["good_return"].append(mean_good)
            history["adv_pg_loss"].append( adv_m.get("pg_loss",    float("nan")))
            history["adv_v_loss"].append(  adv_m.get("value_loss", float("nan")))
            history["adv_entropy"].append( adv_m.get("entropy",    float("nan")))
            history["good_pg_loss"].append(good_m.get("pg_loss",    float("nan")))
            history["good_v_loss"].append( good_m.get("value_loss", float("nan")))
            history["good_entropy"].append(good_m.get("entropy",    float("nan")))

        if update_count % cfg.eval_interval == 0:
            eval_res = evaluate(ippo, cfg)
            print(
                f"  [eval] adv: {eval_res['adversary']['mean']:+.2f}"
                f" ± {eval_res['adversary']['std']:.2f}  |  "
                f"good: {eval_res['good']['mean']:+.2f}"
                f" ± {eval_res['good']['std']:.2f}"
            )

        if update_count % cfg.checkpoint_interval == 0:
            ippo.save(os.path.join(cfg.save_dir, f"ippo_update{update_count}.pt"))

    ippo.save(os.path.join(cfg.save_dir, "ippo_final.pt"))
    env.close()
    print(f"\nTraining complete. Total steps: {total_steps:,}")
    return ippo, history

In [9]:
def evaluate(ippo: "IPPO", cfg: Config, n_episodes: int = None, render: bool = False) -> dict:
    """
    Deterministic evaluation of a trained IPPO agent.

    Uses argmax (greedy) actions — no exploration noise.
    Reports role-separated episode returns.

    Args:
        ippo:       trained IPPO instance
        cfg:        Config (uses max_cycles and eval_episodes)
        n_episodes: number of evaluation episodes (default: cfg.eval_episodes)
        render:     if True, opens a pygame window (render_mode='human')

    Returns:
        {"adversary": {"mean", "std", "min", "max"}, "good": {...}}
    """
    n_episodes  = n_episodes or cfg.eval_episodes
    render_mode = "human" if render else None
    eval_env    = make_env(max_cycles=cfg.max_cycles, render_mode=render_mode)

    roles        = get_agent_roles(eval_env.reset()[0])
    role_returns: Dict[str, list] = {"adversary": [], "good": []}

    for ep in range(n_episodes):
        obs_dict, _ = eval_env.reset()
        ep_ret      = {a: 0.0 for a in eval_env.agents}

        while eval_env.agents:
            actions  = {a: ippo.act_deterministic(a, obs_dict[a]) for a in eval_env.agents}
            obs_dict, rewards, _, _, _ = eval_env.step(actions)
            for a, r in rewards.items():
                ep_ret[a] = ep_ret.get(a, 0.0) + r

        for a, ret in ep_ret.items():
            role_returns[roles[a]].append(ret)

    eval_env.close()

    results = {}
    for role in ("adversary", "good"):
        arr = np.array(role_returns[role])
        results[role] = dict(
            mean=float(arr.mean()), std=float(arr.std()),
            min=float(arr.min()),   max=float(arr.max()),
        )
    return results

In [ ]:
# ── Run training ──────────────────────────────────────────────────────────────
# Adjust total_timesteps for how long you want to train:
#   200_000  — quick sanity check  (~2 min on CPU)
#   1_000_000 — short run to observe early learning
#   3_000_000 — recommended full baseline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

cfg = Config(
    total_timesteps=500_000,
    rollout_steps=2048,
    ppo_epochs=5,
    num_minibatches=1,
    clip_eps=0.1,
    log_interval=5,
    eval_interval=50,
    checkpoint_interval=50,
    eval_episodes=10,
    seed=42,
    device=device,
)

ippo, history = train(cfg)

Using device: cuda
Training IPPO on simple_tag_v3
  adv_obs=16  good_obs=14  actions=5
  total_timesteps=500,000  rollout_steps=2048
  ppo_epochs=5  num_minibatches=1  clip_eps=0.1



In [ ]:
# ── Run deterministic evaluation ──────────────────────────────────────────────

results = evaluate(ippo, cfg, n_episodes=20)

print(f"\n{'Role':>12} | {'Mean':>8} | {'Std':>8} | {'Min':>8} | {'Max':>8}")
print("-" * 54)
for role in ("adversary", "good"):
    r = results[role]
    print(f"{role:>12} | {r['mean']:>+8.2f} | {r['std']:>8.2f} | {r['min']:>+8.2f} | {r['max']:>+8.2f}")

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────────

def plot_training(history: dict):
    """Plot episode returns, policy loss, value loss, and entropy over training."""
    if not history["steps"]:
        print("No data to plot yet.")
        return

    steps = history["steps"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("IPPO Training — simple_tag_v3", fontsize=13)

    ax = axes[0, 0]
    ax.plot(steps, history["adv_return"],  label="adversary", color="tab:red")
    ax.plot(steps, history["good_return"], label="good agent", color="tab:blue")
    ax.set_title("Episode Return (rolling mean)")
    ax.set_xlabel("Env steps")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(steps, history["adv_pg_loss"],  color="tab:red",  label="adversary")
    ax.plot(steps, history["good_pg_loss"], color="tab:blue", label="good agent")
    ax.set_title("Policy (PG) Loss")
    ax.set_xlabel("Env steps")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(steps, history["adv_v_loss"],  color="tab:red",  label="adversary")
    ax.plot(steps, history["good_v_loss"], color="tab:blue", label="good agent")
    ax.set_title("Value Loss")
    ax.set_xlabel("Env steps")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.plot(steps, history["adv_entropy"],  color="tab:red",  label="adversary")
    ax.plot(steps, history["good_entropy"], color="tab:blue", label="good agent")
    ax.set_title("Policy Entropy")
    ax.set_xlabel("Env steps")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out_path = "training_curves.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")


plot_training(history)


# ── Load a saved checkpoint ────────────────────────────────────────────────────
# Uncomment to load and evaluate a previously saved model:
#
# _env_tmp = make_env()
# _obs, _  = _env_tmp.reset()
# _roles_tmp = get_agent_roles(_env_tmp.agents)
# _adv_agents_tmp  = [a for a in _env_tmp.agents if _roles_tmp[a] == "adversary"]
# _good_agents_tmp = [a for a in _env_tmp.agents if _roles_tmp[a] == "good"]
# _adv_obs  = _env_tmp.observation_space(_adv_agents_tmp[0]).shape[0]
# _good_obs = _env_tmp.observation_space(_good_agents_tmp[0]).shape[0]
# _act_dim  = _env_tmp.action_space(_adv_agents_tmp[0]).n
# _env_tmp.close()
#
# cfg_load  = Config()
# ippo_load = IPPO(_adv_agents_tmp, _good_agents_tmp, _adv_obs, _good_obs, _act_dim, cfg_load)
# ippo_load.load("models/ippo_final.pt")
# evaluate(ippo_load, cfg_load, n_episodes=20)